# Six lois de TP retrouvées à partir de dix mesures

En travaux pratiques, on mesure une grandeur en en faisant varier une autre,
puis on cherche la loi qui les relie. Ce notebook confie cette recherche à
GP_ELITE, sur six expériences classiques : le pendule simple, la masse au bout
d'un ressort, la chute libre, la puissance électrique, le gaz parfait et
l'éclairement d'une source ponctuelle.

On ne lui donne pas la forme de la loi. On lui donne dix mesures et les unités
de chaque colonne. Il rend la loi du manuel et, quand une constante physique
manque dans le tableau (g, R, la puissance de la source), ses unités et sa
valeur.

Les mesures sont simulées : la loi du manuel, plus une erreur aléatoire de
2 %, comme celle d'un instrument de TP. On connaît ainsi la bonne réponse. La
dernière section montre comment utiliser vos propres mesures.

**Ce qui a été mesuré.** Dans notre diagnostic, 400 recherches au total, ces
six lois reviennent avec les unités 120 fois sur 120. Il y avait 5 tirages
pour chaque loi, à 10 ou 20 points, avec 2 % ou 5 % d'erreur. Les constantes
déduites, g et R, sont à environ 1 % près en général et à 4,7 % au pire
([le diagnostic](https://github.com/ariel95500-create/gp-elite/blob/main/benchmarks/results_1.1/DIAGNOSTIC_TP.md)).

Exécutez les cellules dans l'ordre avec **Maj + Entrée**. Chaque recherche
prend de trente secondes à une minute environ (de 42 à 55 s sur notre
machine). Les résultats cités sous chaque cellule viennent de notre
exécution de ce notebook avec GP_ELITE 1.1.0 ; sur une autre machine, les
derniers chiffres peuvent différer.

## 1. Installation

In [ ]:
!pip install -q gp-elite sympy

import time
import numpy as np
import gp_elite
from gp_elite import symbolic_regression
print('GP_ELITE', gp_elite.__version__, 'prêt.')

## 2. Deux outils pour la suite

`mesures` simule un tableau de mesures. Pour une seule grandeur variée, les
points sont régulièrement espacés, comme en TP ; pour plusieurs, ils sont
tirés au hasard. Chaque valeur mesurée reçoit une erreur aléatoire de 2 %.

`chercher` lance GP_ELITE avec les unités et affiche la formule trouvée.
`parallel=False` rend le résultat identique d'une exécution à l'autre.

In [ ]:
def mesures(loi, plages, n=10, erreur=0.02, graine=0):
    rng = np.random.default_rng(graine)
    if len(plages) == 1:
        lo, hi = plages[0]
        X = np.linspace(lo, hi, n).reshape(-1, 1)
    else:
        X = np.column_stack([rng.uniform(lo, hi, n) for lo, hi in plages])
    y = loi(X) * (1 + erreur * rng.standard_normal(n))
    return X, y


def chercher(X, y, noms, unites, unite_cible, constante_inconnue):
    t0 = time.time()
    r = symbolic_regression(X, y, feature_names=noms,
                            units=dict(zip(noms, unites)),
                            target_units=unite_cible,
                            unknown_constant=constante_inconnue,
                            parallel=False, seed=0)
    print(f'formule trouvée : {r.expression}   ({time.time() - t0:.0f} s)')
    return r


def coefficient(r):
    # le facteur numérique devant la formule
    return float(r.sympy_expr().as_coeff_Mul()[0])

## 3. Le pendule simple : T = 2π·√(L/g)

On fait varier la longueur L du fil, de 10 cm à 1 m, et on mesure la période T.
g n'est pas dans le tableau : `constante_inconnue=True` laisse GP_ELITE
déduire les unités et la valeur de la constante qui manque.

In [ ]:
X, T = mesures(lambda X: 2 * np.pi * np.sqrt(X[:, 0] / 9.81), [(0.10, 1.00)])
r = chercher(X, T, ['L'], ['m'], 's', constante_inconnue=True)
c = coefficient(r)
print(f'T = {c:.4f} * sqrt(L), donc g = (2*pi/{c:.4f})^2 = {(2 * np.pi / c) ** 2:.2f} m/s^2')

Sur notre exécution (GP_ELITE 1.1.0) : `2.00564 * sqrt(L)` en 42 s, la forme de la loi du manuel. On en déduit g = 9,81 m/s², à moins de 0,1 % de la valeur exacte.

## 4. La masse au bout d'un ressort : T = 2π·√(m/k)

Deux grandeurs varient : la masse m (kg) et la raideur k (N/m, soit kg/s²).
Aucune constante physique ne manque, seulement le facteur 2π, sans unité.

In [ ]:
X, T = mesures(lambda X: 2 * np.pi * np.sqrt(X[:, 0] / X[:, 1]),
               [(0.05, 0.50), (10.0, 50.0)])
r = chercher(X, T, ['m', 'k'], ['kg', 'kg/s^2'], 's', constante_inconnue=False)
print(f'facteur devant la racine : {coefficient(r):.4f}  (2*pi = {2 * np.pi:.4f})')

Sur notre exécution : `6.30042 * sqrt(m / k)` en 51 s. Le facteur vaut 6,300, pour 2π = 6,283 : 0,3 % d'écart, dû à l'erreur des mesures.

## 5. La chute libre : d = ½·g·t²

On mesure la distance parcourue d après une durée t, de 0,1 à 0,6 s.

In [ ]:
X, d = mesures(lambda X: 0.5 * 9.81 * X[:, 0] ** 2, [(0.10, 0.60)])
r = chercher(X, d, ['t'], ['s'], 'm', constante_inconnue=True)
c = coefficient(r)
print(f'd = {c:.4f} * t^2, donc g = 2 * {c:.4f} = {2 * c:.2f} m/s^2')

Sur notre exécution : `4.87056 * t²` en 55 s, d'où g = 9,74 m/s², 0,7 % sous 9,81.

## 6. La puissance électrique : P = U²/R

La tension U (V) et la résistance R (Ω) varient. Les unités électriques
s'écrivent en unités de base : V = kg·m²·s⁻³·A⁻¹ et Ω = kg·m²·s⁻³·A⁻².

In [ ]:
X, P = mesures(lambda X: X[:, 0] ** 2 / X[:, 1], [(1.0, 12.0), (10.0, 100.0)])
r = chercher(X, P, ['U', 'R'], ['kg*m^2/(s^3*A)', 'kg*m^2/(s^3*A^2)'],
             'kg*m^2/s^3', constante_inconnue=False)

Sur notre exécution : `1.00285 * U² / R` en 47 s : la loi P = U²/R, avec un facteur 1,003 qui vient de l'erreur des mesures.

## 7. Le gaz parfait : P = n·R·T/V

La quantité de matière n (mol), la température T (K) et le volume V (m³)
varient. La constante des gaz parfaits R n'est pas dans le tableau.

In [ ]:
X, P = mesures(lambda X: X[:, 0] * 8.314 * X[:, 1] / X[:, 2],
               [(0.01, 0.10), (273.0, 373.0), (1e-3, 5e-3)])
r = chercher(X, P, ['n', 'T', 'V'], ['mol', 'K', 'm^3'], 'kg/(m*s^2)',
             constante_inconnue=True)
print(f'R = {coefficient(r):.3f} J/(mol K)  (valeur exacte 8.314)')

Sur notre exécution : `8.24992 * n * T / V` en 53 s, d'où R = 8,25 J/(mol·K), 0,8 % sous 8,314.

## 8. L'éclairement d'une source ponctuelle : E = P/(4π·d²)

On mesure l'éclairement E (W/m², soit kg/s³) à une distance d de la source,
de 20 cm à 2 m. La puissance P de la source n'est pas dans le tableau.

In [ ]:
X, E = mesures(lambda X: 10.0 / (4 * np.pi * X[:, 0] ** 2), [(0.2, 2.0)])
r = chercher(X, E, ['d'], ['m'], 'kg/s^3', constante_inconnue=True)
c = coefficient(r)
print(f'E = {c:.4f} / d^2, donc P = 4*pi*{c:.4f} = {4 * np.pi * c:.2f} W  (valeur exacte 10 W)')

Sur notre exécution : `0.797624 * (d / d²)²` en 46 s. C'est bien 0,7976/d², mais écrit maladroitement : (d/d²)² vaut 1/d². On en déduit P = 10,02 W pour 10 W.

## 9. Avec vos propres mesures

Remplacez les listes par vos valeurs, et les unités par celles de vos
colonnes. Les unités s'écrivent en unités de base du Système international
(kg, m, s, A, K, mol), par exemple `'m/s^2'` ou `'kg*m^2/s^2'`. Si une
constante physique de la loi n'est pas dans votre tableau, gardez
`constante_inconnue=True`.

```python
L = [0.20, 0.35, 0.50, 0.65, 0.80]          # longueur du fil, en m
T = [0.90, 1.19, 1.42, 1.62, 1.80]          # période mesurée, en s
r = chercher(np.array(L).reshape(-1, 1), np.array(T), ['L'], ['m'], 's',
             constante_inconnue=True)
```

## 10. Ce que GP_ELITE ne sait pas encore faire en TP

Notre diagnostic l'a mesuré ([le détail](https://github.com/ariel95500-create/gp-elite/blob/main/benchmarks/results_1.1/DIAGNOSTIC_TP.md)) :

- **Une constante avec unités à l'intérieur d'une fonction ou d'une somme.**
  Avec les unités, la décharge d'un condensateur u = E·exp(−t/τ), le
  refroidissement de Newton et la relation de conjugaison d'une lentille ne
  peuvent pas être construits : τ, la température ambiante ou la distance
  focale ont des unités. L'outil rend alors une formule médiocre, sans le
  dire. Pour ces lois, lancez la recherche sans les unités : la décharge du
  condensateur revient alors 4 fois sur 5 avec 10 points, mais elle est
  écrite comme une puissance d'un nombre (par exemple 0,0078^(0,2·t)) plutôt
  que comme une exponentielle de −t/τ.
- **La loi de Snell-Descartes**, qui demande un arc sinus, que GP_ELITE ne
  propose pas.
- **La durée.** Une recherche prend de trente secondes à une minute et
  demie selon la machine et la loi, ce qui reste long devant une classe.
  Avec des mesures, qui ont toujours une erreur, aucune loi n'est exacte :
  la recherche va au bout de son budget, même quand la loi du manuel est
  dans la population de départ (depuis la version 1.1, c'est le cas par
  exemple pour la chute libre et le gaz parfait).
- **Plusieurs formules au choix.** En dessous de 30 mesures, GP_ELITE rend
  une seule formule, sans liste de formules candidates.

Ces points sont le programme de la prochaine version.